In [65]:
import numpy as np

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(precision=3, suppress=True)

In [222]:
def _obstacle_coords_(obstacle_coords, n_rows, n_cols, L_x, L_y):
    # Obstacle coordinates in real-world space
    (x1_real, y1_real), (x2_real, y2_real) = obstacle_coords
    
    x1 = x1_real/L_x
    x2 = x2_real/L_x
    y1 = y1_real/L_y
    y2 = y2_real/L_y
    # Map real-world coordinates to grid indices
    pixel_x1 = int(x1 * n_cols)  + 1
    pixel_y1 = (n_rows - 1) - int(y2*n_rows) + 1
    pixel_x2 = int(x2 * n_cols)  + 1
    pixel_y2 = (n_rows - 1) - int(y1*n_rows) + 1
    
    return (pixel_x1, pixel_y1), (pixel_x2, pixel_y2)

In [209]:
def create_obstacle_mask(obstacle_coords, n_rows, n_cols, L_x, L_y):
    # Obstacle coordinates in real-world space
    (x1_real, y1_real), (x2_real, y2_real) = obstacle_coords
    
    x1 = x1_real/L_x
    x2 = x2_real/L_x
    y1 = y1_real/L_y
    y2 = y2_real/L_y
    # Map real-world coordinates to grid indices
    pixel_x1 = int(x1 * n_cols)  + 1
    pixel_y1 = (n_rows - 1) - int(y2*n_rows) + 1
    pixel_x2 = int(x2 * n_cols)  + 1
    pixel_y2 = (n_rows - 1) - int(y1*n_rows) + 1
    
    # Step 2: Create the obstacle mask (0 = empty, 1 = obstacle)
    mask = np.zeros((n_rows+2, n_cols+2), dtype=int)
    
    # Set the obstacle region (from x1 to x2, and y1 to y2) to 1
    mask[pixel_y1:pixel_y2+1, pixel_x1:pixel_x2 + 1] = 1
    
    return mask

In [227]:
n_rows, n_cols = 6, 6  # 10x10 grid
L_x, L_y = 1, 1    # Grid size in real-world coordinates (10 by 10 units)
obstacle_coords = _obstacle_coords_(((0, 0), (1/3, 1/3)), n_cols, n_rows, L_x, L_y)  # Real-world coordinates of obstacle

(x1, y1), (x2, y2) = obstacle_coords

In [373]:
cells = np.ones((8,8))
cells[:, [0,-1]] = 0
cells[[0,-1], :] = 0
cells[0, 1:-1] = 1
cells[y1:y2+1, x1:x2+1] = 0

In [378]:
cells

array([[0., 1., 1., 1., 1., 1., 1., 0.],
       [0., 1., 1., 1., 1., 1., 1., 0.],
       [0., 1., 1., 1., 1., 1., 1., 0.],
       [0., 1., 1., 1., 1., 1., 1., 0.],
       [0., 0., 0., 0., 1., 1., 1., 0.],
       [0., 0., 0., 0., 1., 1., 1., 0.],
       [0., 0., 0., 0., 1., 1., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.]])

In [376]:

u_f = ((cells[:, :-1] == 0) | (cells[:, 1:] == 0))


In [377]:
u_f

array([[ True, False, False, False, False, False,  True],
       [ True, False, False, False, False, False,  True],
       [ True, False, False, False, False, False,  True],
       [ True, False, False, False, False, False,  True],
       [ True,  True,  True,  True, False, False,  True],
       [ True,  True,  True,  True, False, False,  True],
       [ True,  True,  True,  True, False, False,  True],
       [ True,  True,  True,  True,  True,  True,  True]])

In [379]:
v_f = ((cells[:-1, :] == 0) | (cells[1:, :] == 0))


In [380]:
v_f

array([[ True, False, False, False, False, False, False,  True],
       [ True, False, False, False, False, False, False,  True],
       [ True, False, False, False, False, False, False,  True],
       [ True,  True,  True,  True, False, False, False,  True],
       [ True,  True,  True,  True, False, False, False,  True],
       [ True,  True,  True,  True, False, False, False,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True]])

In [103]:
mask

array([[False, False, False, False, False, False, False, False],
       [False, False, False,  True,  True,  True, False, False],
       [False, False,  True,  True,  True,  True,  True, False],
       [False, False,  True,  True,  True,  True,  True, False],
       [False, False,  True,  True,  True,  True,  True, False],
       [False, False,  True,  True,  True,  True,  True, False],
       [False, False, False,  True,  True, False, False, False],
       [False, False, False, False, False, False, False, False]])

In [132]:
from scipy.ndimage import convolve


mask = np.zeros((n_rows, n_cols))  # Assuming interior = 0
mask[0, :] = 1  # Top boundary as solid
mask[:, 0] = 1  # Left boundary as solid
mask[-1, :] = 1  # Bottom boundary as solid
mask[:, -1] = 1  # Right boundary as solid

# Define a simple convolution kernel to detect edges
kernel = np.array([[1, 1, 0],
                   [1, 1, 0],
                   [1, 1, 0]])

# Convolve the mask with the kernel to detect edges
edge_mask = convolve(mask, kernel, mode='constant', cval=0)

# Any positive value in edge_mask corresponds to a border
edge_mask = edge_mask > 0


In [133]:
edge_mask

array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True, False, False, False, False, False, False, False,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True]])